# Adversarial Attack Projesi
**Deep Learning Donem Projesi**

| | |
|---|---|
| Konu | CNN saglalmliginin FGSM, PGD saldirilarindan analizi ve adversarial training ile savunma |
| Veri Seti | CIFAR-10 |
| Framework | PyTorch |

---
## Icerik
1. Kurulum ve Hazirlik
2. Veri Yukleme
3. CNN Modeli
4. Model Egitimi
5. FGSM Saldirisi
6. PGD Saldirisi
7. Adversarial Training (Savunma)
8. Black-box Transferability
9. Gorsellestirme ve Analiz

---
## 1. Kurulum ve Hazirlik

In [ ]:
# Gerekli kutuphaneleri yukle
!pip install torchattacks -q

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import torchattacks
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

print('Kutuphaneler yuklendi.')

In [ ]:
# GPU kontrolu
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Kullanilan cihaz: {device}')

if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('UYARI: GPU bulunamadi. Calisma zamani cok uzun surebilir.')
    print('Colab menusu: Calisma Zamani > Calisma Zamani Turunu Degistir > T4 GPU')

In [ ]:
# Google Drive baglantisi (modelleri kaydetmek icin)
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/adversarial_project/'

import os
os.makedirs(DRIVE_PATH, exist_ok=True)
print(f'Drive baglandi: {DRIVE_PATH}')

In [ ]:
# Reproducibility - her calistirmada ayni sonuclar
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# Proje genelinde kullanilacak sabitler
BATCH_SIZE  = 128
NUM_EPOCHS  = 20
LEARNING_RATE = 0.001
EPSILONS    = [0, 0.01, 0.05, 0.1, 0.2, 0.3]  # saldiri gucleri

CLASSES = ('plane','car','bird','cat','deer',
           'dog','frog','horse','ship','truck')

print('Sabitler tanimlandi.')
print(f'  Batch size  : {BATCH_SIZE}')
print(f'  Epoch sayisi: {NUM_EPOCHS}')
print(f'  Epsilon list: {EPSILONS}')

---
## 2. Veri Yukleme

In [ ]:
# Veri on isleme
# Egitim: veri artirma + normalizasyon
# Test  : sadece normalizasyon
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),   # rastgele yatay cevir
    transforms.RandomCrop(32, padding=4),# rastgele kirp
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std =(0.2023, 0.1994, 0.2010)
    )
])

transform_test = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std =(0.2023, 0.1994, 0.2010)
    )
])

print('Transformlar tanimlandi.')

In [ ]:
# CIFAR-10 indir ve yukle
train_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=True,  download=True, transform=transform_train
)
test_dataset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test
)

train_loader = torch.utils.data.DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

print(f'Egitim seti : {len(train_dataset):,} goruntu')
print(f'Test seti   : {len(test_dataset):,} goruntu')
print(f'Sinif sayisi: {len(CLASSES)}')

In [ ]:
# Veri setinden ornek gorseller
def show_sample_images(loader, classes, n=10):
    images, labels = next(iter(loader))
    fig, axes = plt.subplots(1, n, figsize=(15, 2))
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3,1,1)
    for i in range(n):
        img = images[i] * std + mean          # normalizasyonu geri al
        img = img.permute(1,2,0).numpy()
        img = np.clip(img, 0, 1)
        axes[i].imshow(img)
        axes[i].set_title(classes[labels[i]], fontsize=8)
        axes[i].axis('off')
    plt.suptitle('CIFAR-10 Ornek Gorseller', fontsize=11)
    plt.tight_layout()
    plt.show()

show_sample_images(test_loader, CLASSES)

---
## 3. CNN Modeli

In [ ]:
class SimpleCNN(nn.Module):
    """Temel CNN modeli - Model A olarak kullanilacak"""
    def __init__(self):
        super().__init__()
        # Ozellik cikarici
        self.features = nn.Sequential(
            # Blok 1: 3->32, 32x32->16x16
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
            # Blok 2: 32->64, 16x16->8x8
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.25),
        )
        # Siniflandirici
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 8 * 8, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class DeeperCNN(nn.Module):
    """Daha derin CNN - Model B olarak kullanilacak (transferability icin)"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            # Blok 1: 3->64
            nn.Conv2d(3, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Blok 2: 64->128
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            # Blok 3: 128->256
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


# Model ozeti
model_a = SimpleCNN().to(device)
model_b = DeeperCNN().to(device)

total_a = sum(p.numel() for p in model_a.parameters())
total_b = sum(p.numel() for p in model_b.parameters())
print(f'Model A (SimpleCNN) - Parametre sayisi: {total_a:,}')
print(f'Model B (DeeperCNN) - Parametre sayisi: {total_b:,}')

---
## 4. Model Egitimi

In [ ]:
def train_model(model, train_loader, test_loader,
                num_epochs, lr, device, save_path=None):
    """Standart egitim dongusu."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)

    history = {'train_loss': [], 'test_acc': []}

    for epoch in range(num_epochs):
        # --- Egitim ---
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        # --- Test ---
        acc = evaluate_model(model, test_loader, device)
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['test_acc'].append(acc)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1:2d}/{num_epochs}] '
                  f'Loss: {avg_loss:.4f} | '
                  f'Test Acc: {acc:.2f}%')

    # Modeli kaydet
    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f'Model kaydedildi: {save_path}')

    return history


def evaluate_model(model, loader, device):
    """Test seti dogrulugunu hesapla."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            _, predicted = torch.max(model(images), 1)
            correct += (predicted == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

print('Egitim fonksiyonlari tanimlandi.')

In [ ]:
# Model A egit
print('=== Model A (SimpleCNN) egitiliyor ===')
history_a = train_model(
    model_a, train_loader, test_loader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    device=device,
    save_path=DRIVE_PATH + 'model_a.pth'
)
print(f'Model A final dogrulugu: {history_a["test_acc"][-1]:.2f}%')

In [ ]:
# Model B egit
print('=== Model B (DeeperCNN) egitiliyor ===')
history_b = train_model(
    model_b, train_loader, test_loader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    device=device,
    save_path=DRIVE_PATH + 'model_b.pth'
)
print(f'Model B final dogrulugu: {history_b["test_acc"][-1]:.2f}%')

In [ ]:
# Mevcut modeli yukle (sifirdan egitmemek icin)
# model_a.load_state_dict(torch.load(DRIVE_PATH + 'model_a.pth'))
# model_b.load_state_dict(torch.load(DRIVE_PATH + 'model_b.pth'))
# print('Modeller Drive dan yuklendi.')

---
## 5. FGSM Saldirisi

In [ ]:
def fgsm_attack(model, images, labels, epsilon, criterion):
    """FGSM - Fast Gradient Sign Method.
    Formul: x_adv = x + epsilon * sign(grad_x Loss)
    """
    images = images.clone().detach().requires_grad_(True)
    loss   = criterion(model(images), labels)
    model.zero_grad()
    loss.backward()
    adv = images + epsilon * images.grad.sign()
    return torch.clamp(adv, 0, 1).detach()


def test_fgsm(model, loader, epsilon, device):
    """Belirli epsilon degerinde FGSM saldirisi altinda dogruluk."""
    criterion = nn.CrossEntropyLoss()
    model.eval()
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if epsilon > 0:
            images = fgsm_attack(model, images, labels, epsilon, criterion)
        with torch.no_grad():
            _, pred = torch.max(model(images), 1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

print('FGSM fonksiyonlari tanimlandi.')

In [ ]:
# Farkli epsilon degerlerinde FGSM testi
print('FGSM saldirisi test ediliyor...')
fgsm_results = []
for eps in EPSILONS:
    acc = test_fgsm(model_a, test_loader, eps, device)
    fgsm_results.append(acc)
    print(f'  epsilon={eps:.2f} --> Dogruluk: {acc:.2f}%')

---
## 6. PGD Saldirisi

In [ ]:
def test_pgd(model, loader, epsilon, alpha=None,
             steps=40, device='cuda'):
    """torchattacks ile PGD saldirisi.
    alpha: adim buyuklugu (varsayilan: epsilon/4)
    steps: iterasyon sayisi
    """
    if alpha is None:
        alpha = epsilon / 4

    model.eval()
    atk = torchattacks.PGD(model, eps=epsilon,
                            alpha=alpha, steps=steps)
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if epsilon > 0:
            images = atk(images, labels)
        with torch.no_grad():
            _, pred = torch.max(model(images), 1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

print('PGD fonksiyonu tanimlandi.')

In [ ]:
# Farkli epsilon degerlerinde PGD testi
print('PGD saldirisi test ediliyor...')
pgd_results = []
for eps in EPSILONS:
    acc = test_pgd(model_a, test_loader, eps, device=device)
    pgd_results.append(acc)
    print(f'  epsilon={eps:.2f} --> Dogruluk: {acc:.2f}%')

---
## 7. Adversarial Training (Savunma)

In [ ]:
def adversarial_train(model, train_loader, test_loader,
                      num_epochs, lr, epsilon, device, save_path=None):
    """Adversarial training - modeli saldirili orneklerle egit."""
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)
    history   = {'train_loss': [], 'test_acc': []}

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0
        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            # Saldirili ornek uret
            adv_images = fgsm_attack(model, images, labels,
                                     epsilon, criterion)
            # Saldirili ornekle egit
            optimizer.zero_grad()
            loss = criterion(model(adv_images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        acc      = evaluate_model(model, test_loader, device)
        avg_loss = total_loss / len(train_loader)
        history['train_loss'].append(avg_loss)
        history['test_acc'].append(acc)

        if (epoch + 1) % 5 == 0:
            print(f'Epoch [{epoch+1:2d}/{num_epochs}] '
                  f'Adv Loss: {avg_loss:.4f} | '
                  f'Clean Acc: {acc:.2f}%')

    if save_path:
        torch.save(model.state_dict(), save_path)
        print(f'Savunmali model kaydedildi: {save_path}')

    return history

print('Adversarial training fonksiyonu tanimlandi.')

In [ ]:
# Savunmali modeli egit
robust_model = SimpleCNN().to(device)

print('=== Savunmali Model egitiliyor ===')
history_robust = adversarial_train(
    robust_model, train_loader, test_loader,
    num_epochs=NUM_EPOCHS,
    lr=LEARNING_RATE,
    epsilon=0.1,
    device=device,
    save_path=DRIVE_PATH + 'robust_model.pth'
)
print(f'Savunmali model clean dogrulugu: {history_robust["test_acc"][-1]:.2f}%')

In [ ]:
# Savunmali modeli FGSM ve PGD ile test et
print('Savunmali model saldiri altinda test ediliyor...')
robust_fgsm = []
robust_pgd  = []
for eps in EPSILONS:
    acc_f = test_fgsm(robust_model, test_loader, eps, device)
    acc_p = test_pgd(robust_model,  test_loader, eps, device=device)
    robust_fgsm.append(acc_f)
    robust_pgd.append(acc_p)
    print(f'  epsilon={eps:.2f} | FGSM: {acc_f:.2f}% | PGD: {acc_p:.2f}%')

---
## 8. Black-box Transferability

In [ ]:
def transferability_test(source_model, target_model,
                          loader, epsilon, device):
    """Black-box transferability testi.
    Kaynak modelde uretilen saldirilar hedef modele transfer olur mu?
    """
    criterion = nn.CrossEntropyLoss()
    source_model.eval()
    target_model.eval()

    correct_clean = correct_adv = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Kaynak modelde saldiri uret
        adv_images = fgsm_attack(
            source_model, images, labels, epsilon, criterion
        )

        # Hedef modelde test et (gradyana dokunmuyoruz)
        with torch.no_grad():
            _, pred_c = torch.max(target_model(images), 1)
            _, pred_a = torch.max(target_model(adv_images), 1)
            correct_clean += (pred_c == labels).sum().item()
            correct_adv   += (pred_a == labels).sum().item()
            total         += labels.size(0)

    clean_acc    = 100.0 * correct_clean / total
    adv_acc      = 100.0 * correct_adv   / total
    transfer_rate = clean_acc - adv_acc
    return clean_acc, adv_acc, transfer_rate

print('Transferability fonksiyonu tanimlandi.')

In [ ]:
# 3 senaryo karsilastirmasi
scenarios = [
    ('White-box (A -> A)',       model_a, model_a),
    ('Black-box (A -> B)',       model_a, model_b),
    ('Black-box + Savunma (A -> Robust)', model_a, robust_model),
]

print(f'Transferability testi (epsilon=0.1)\n')
print(f'{'Senaryo':<35} {'Temiz':>8} {'Saldiri':>10} {'Etki':>8}')
print('-' * 65)

transfer_results = []
for name, src, tgt in scenarios:
    clean, adv, rate = transferability_test(
        src, tgt, test_loader, epsilon=0.1, device=device
    )
    transfer_results.append((name, clean, adv, rate))
    print(f'{name:<35} {clean:>7.2f}% {adv:>9.2f}% {rate:>7.2f}%')

---
## 6.5  C&W Saldirisi (Carlini & Wagner)

FGSM ve PGD'den farkli olarak C&W, perturbasyonu minimize etmeyi hedefler: modeli yaniltmak icin **en az bozulmayi** bul. Matematiksel olarak bir optimizasyon problemi olarak kurgulanir.

**Neden ekledik?** FGSM ve PGD gradient-sign tabanli. C&W ise farkli bir paradigma — uc saldiriyi karsilastirmak projeye gercek analitik derinlik katıyor.

In [ ]:
def test_cw(model, loader, device, c=1, kappa=0, steps=100, lr=0.01):
    """torchattacks ile C&W saldirisi.
    c     : saldiri gucu (yuksek = daha agresif)
    kappa : confidence margin
    steps : optimizasyon adimi
    lr    : optimizer learning rate
    """
    model.eval()
    atk = torchattacks.CW(model, c=c, kappa=kappa, steps=steps, lr=lr)
    correct = total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        adv = atk(images, labels)
        with torch.no_grad():
            _, pred = torch.max(model(adv), 1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)
    return 100.0 * correct / total

print('C&W fonksiyonu tanimlandi.')
print('NOT: C&W yavas calisir -- test loader yerine kucuk bir subset kullaniyoruz.')

In [ ]:
# C&W cok yavas oldugu icin kucuk subset kullan
subset = torch.utils.data.Subset(test_dataset, range(500))
subset_loader = torch.utils.data.DataLoader(
    subset, batch_size=32, shuffle=False
)

print('C&W saldirisi test ediliyor (500 ornek)...')
cw_acc = test_cw(model_a, subset_loader, device)
print(f'C&W saldirisi altinda dogruluk: {cw_acc:.2f}%')

# Karsilastirma icin FGSM ve PGD'yi de ayni subset uzerinde test et
fgsm_sub = test_fgsm(model_a, subset_loader, epsilon=0.1, device=device)
pgd_sub  = test_pgd(model_a,  subset_loader, epsilon=0.1, device=device)

print(f'\nAyni 500 ornek uzerinde karsilastirma:')
print(f'  FGSM (eps=0.1) : {fgsm_sub:.2f}%')
print(f'  PGD  (eps=0.1) : {pgd_sub:.2f}%')
print(f'  C&W            : {cw_acc:.2f}%')

In [ ]:
# Uc saldiriyi bar chart ile karsilastir
attack_names = ['Temiz', 'FGSM', 'PGD', 'C&W']
clean_sub = evaluate_model(model_a, subset_loader, device)
attack_accs = [clean_sub, fgsm_sub, pgd_sub, cw_acc]
colors = ['steelblue', 'orange', 'tomato', 'purple']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(attack_names, attack_accs, color=colors, alpha=0.85, width=0.5)

for bar, acc in zip(bars, attack_accs):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1,
            f'{acc:.1f}%', ha='center', fontsize=11, fontweight='bold')

ax.set_ylabel('Dogruluk (%)', fontsize=12)
ax.set_title('Saldiri Yontemleri Karsilastirmasi\n(500 test ornegi, epsilon=0.1)',
             fontsize=12, fontweight='bold')
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')
ax.axhline(y=10, color='red', linestyle='--', alpha=0.4, label='Rastgele tahmin (%10)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'attack_comparison.png', dpi=150)
plt.show()

print('\nAnaliz:')
print(f'En guclu saldiri  : {attack_names[attack_accs.index(min(attack_accs))]}')
print(f'En zayif saldiri  : {attack_names[1:][([attack_accs[1],attack_accs[2],attack_accs[3]]).index(max([attack_accs[1],attack_accs[2],attack_accs[3]]))]}')

---
## 7.5  Savunma Yontemleri Karsilastirmasi

Sadece adversarial training degil, **uc farkli savunma** yontemini karsilastiriyoruz:

| Savunma | Fikir | Zorluk |
|---|---|---|
| Adversarial Training | Saldirili orneklerle egit | Yuksek |
| Input Preprocessing | Girdiyi bulaniklastir, gurultuyu azalt | Dusuk |
| Feature Squeezing | Renk derinligini azalt | Dusuk |

**Neden onemli?** Her savunmanin farkli tradeoff'lari var. Hangisi ne zaman ise yarar?

In [ ]:
import torch.nn.functional as F

# --- Savunma 1: Input Preprocessing (Gaussian Blur) ---
def gaussian_blur_defense(images, kernel_size=3, sigma=1.0):
    """Goruntu uzerine Gaussian blur uygula.
    Fikir: adversarial perturbasyonlar yuksek frekansli sinyal.
    Blur bu sinyali yumusatir.
    """
    # PyTorch ile blur: unfold + agirlikli ortalama
    channels = images.shape[1]
    # 1D Gaussian kernel olustur
    coords = torch.arange(kernel_size, dtype=torch.float32) - kernel_size // 2
    kernel_1d = torch.exp(-coords**2 / (2 * sigma**2))
    kernel_1d = kernel_1d / kernel_1d.sum()
    kernel_2d = kernel_1d.outer(kernel_1d)
    kernel_2d = kernel_2d.expand(channels, 1, kernel_size, kernel_size).to(images.device)
    padding   = kernel_size // 2
    blurred   = F.conv2d(images, kernel_2d, padding=padding, groups=channels)
    return blurred


# --- Savunma 2: Feature Squeezing ---
def feature_squeezing_defense(images, bit_depth=4):
    """Renk derinligini azalt.
    Fikir: adversarial perturbasyonlar ince piksel farklarinda gizlenir.
    Bit depth azalinca bu farklar kaybolur.
    bit_depth=4 --> her kanal 0-15 araligina indirilir (256'dan 16'ya)
    """
    max_val = 2 ** bit_depth - 1
    squeezed = torch.round(images * max_val) / max_val
    return squeezed


# --- Test fonksiyonu (tum savunmalar icin) ---
def test_with_defense(model, loader, epsilon, device,
                      defense_fn=None, defense_name='Yok'):
    """Savunma uygulayarak saldiri altinda test.
    defense_fn: None ise savunma yok, yoksa fonksiyon uygula.
    """
    criterion = nn.CrossEntropyLoss()
    model.eval()
    correct = total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        # Saldiri uret
        if epsilon > 0:
            adv = fgsm_attack(model, images, labels, epsilon, criterion)
        else:
            adv = images

        # Savunma uygula
        if defense_fn is not None:
            adv = defense_fn(adv)

        with torch.no_grad():
            _, pred = torch.max(model(adv), 1)
            correct += (pred == labels).sum().item()
            total   += labels.size(0)

    return 100.0 * correct / total


print('Savunma fonksiyonlari tanimlandi:')
print('  1. Gaussian Blur  (Input Preprocessing)')
print('  2. Feature Squeezing')
print('  3. Adversarial Training (zaten egitildi)')

In [ ]:
# Tum epsilon degerleri icin uc savunmayi karsilastir
print('Savunma yontemleri karsilastirilıyor...\n')

results_defense = {
    'Savunmasiz'          : [],
    'Gaussian Blur'       : [],
    'Feature Squeezing'   : [],
    'Adv. Training'       : [],
}

for eps in EPSILONS:
    # Savunmasiz model
    acc_none  = test_with_defense(model_a, test_loader, eps, device)
    # Gaussian blur savunmasi
    acc_blur  = test_with_defense(model_a, test_loader, eps, device,
                                   defense_fn=gaussian_blur_defense)
    # Feature squeezing savunmasi
    acc_sq    = test_with_defense(model_a, test_loader, eps, device,
                                   defense_fn=feature_squeezing_defense)
    # Adversarial training (robust_model)
    acc_adv   = test_with_defense(robust_model, test_loader, eps, device)

    results_defense['Savunmasiz'].append(acc_none)
    results_defense['Gaussian Blur'].append(acc_blur)
    results_defense['Feature Squeezing'].append(acc_sq)
    results_defense['Adv. Training'].append(acc_adv)

    print(f'eps={eps:.2f} | '
          f'Savunmasiz: {acc_none:.1f}% | '
          f'Blur: {acc_blur:.1f}% | '
          f'Squeeze: {acc_sq:.1f}% | '
          f'AdvTrain: {acc_adv:.1f}%')

In [ ]:
# Savunma karsilastirma grafigi
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

colors_def = {
    'Savunmasiz'        : ('red',    'o', '-'),
    'Gaussian Blur'     : ('blue',   's', '--'),
    'Feature Squeezing' : ('orange', '^', '-.'),
    'Adv. Training'     : ('green',  'D', '-'),
}

# Sol: Tum epsilon degerleri
for name, accs in results_defense.items():
    c, m, ls = colors_def[name]
    axes[0].plot(EPSILONS, accs, color=c, marker=m,
                 linestyle=ls, linewidth=2, markersize=7, label=name)

axes[0].set_title('Savunma Yontemleri Karsilastirmasi\n(FGSM Saldirisi Altinda)',
                   fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epsilon', fontsize=11)
axes[0].set_ylabel('Dogruluk (%)', fontsize=11)
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# Sag: eps=0.1 icin bar chart (temiz vs saldiri)
eps_idx   = EPSILONS.index(0.1)
defenses  = list(results_defense.keys())
clean_acc = results_defense['Savunmasiz'][0]   # epsilon=0
adv_accs  = [results_defense[d][eps_idx] for d in defenses]
x = np.arange(len(defenses))
width = 0.35

bars1 = axes[1].bar(x - width/2, [clean_acc]*len(defenses),
                     width, label='Temiz Dogruluk',
                     color='steelblue', alpha=0.7)
bars2 = axes[1].bar(x + width/2, adv_accs,
                     width, label='Saldiri Altinda (eps=0.1)',
                     color=[colors_def[d][0] for d in defenses], alpha=0.85)

for bar in bars2:
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1,
                 f'{bar.get_height():.1f}%',
                 ha='center', fontsize=9, fontweight='bold')

axes[1].set_title('Epsilon=0.1 Detay Karsilastirma', fontsize=11, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(defenses, fontsize=9, rotation=10)
axes[1].set_ylabel('Dogruluk (%)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].set_ylim(0, 100)
axes[1].grid(True, alpha=0.3, axis='y')

plt.suptitle('Savunma Yontemleri Analizi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'defense_comparison.png', dpi=150)
plt.show()

In [ ]:
# Savunma tradeoff analizi -- temiz dogruluk ne kadar etkilendi?
print('Savunma Tradeoff Analizi')
print('=' * 55)
print(f'{'Savunma':<22} {'Temiz Acc':>10} {'eps=0.1 Acc':>12} {'Tradeoff':>10}')
print('-' * 55)

baseline_clean = results_defense['Savunmasiz'][0]
for name, accs in results_defense.items():
    clean   = accs[0]                           # epsilon=0
    adv     = accs[EPSILONS.index(0.1)]         # epsilon=0.1
    tradeoff = baseline_clean - clean           # temiz dogruluk kaybi
    print(f'{name:<22} {clean:>9.2f}% {adv:>11.2f}% {tradeoff:>+9.2f}%')

print('=' * 55)
print('\nYorum:')
print('Tradeoff: Savunmanin temiz dogruluga maliyeti')
print('  (+) = temiz dogruluk dustu')
print('  (-) = temiz dogruluk artti (beklenmedik)')
print('\nIdeal savunma: yuksek saldiri direnci, dusuk tradeoff')

---
## 9. Gorsellestirme ve Analiz

In [ ]:
# Grafik 1: Epsilon vs Dogruluk
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Sol: Normal model
axes[0].plot(EPSILONS, fgsm_results, 'r-o', linewidth=2,
             markersize=7, label='FGSM')
axes[0].plot(EPSILONS, pgd_results,  'b-s', linewidth=2,
             markersize=7, label='PGD')
axes[0].set_title('Normal Model — Saldiri Altinda', fontsize=12, fontweight='bold')
axes[0].set_xlabel('Epsilon', fontsize=11)
axes[0].set_ylabel('Test Dogrulugu (%)', fontsize=11)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim(0, 100)

# Sag: Normal vs Savunmali
axes[1].plot(EPSILONS, fgsm_results,  'r-o',  linewidth=2,
             markersize=7, label='Normal (FGSM)')
axes[1].plot(EPSILONS, robust_fgsm,   'g-o',  linewidth=2,
             markersize=7, label='Savunmali (FGSM)')
axes[1].plot(EPSILONS, pgd_results,   'r--s', linewidth=2,
             markersize=7, label='Normal (PGD)')
axes[1].plot(EPSILONS, robust_pgd,    'g--s', linewidth=2,
             markersize=7, label='Savunmali (PGD)')
axes[1].set_title('Normal vs Savunmali Model', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Epsilon', fontsize=11)
axes[1].set_ylabel('Test Dogrulugu (%)', fontsize=11)
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 100)

plt.suptitle('FGSM ve PGD Saldirisi Altinda Model Performansi',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'epsilon_vs_accuracy.png', dpi=150)
plt.show()

In [ ]:
# Grafik 2: Orjinal vs Perturbation vs Saldirili gorsel karsilastirma
def visualize_attacks(model, loader, epsilon, classes, device, n=8):
    criterion = nn.CrossEntropyLoss()
    images, labels = next(iter(loader))
    images, labels = images[:n].to(device), labels[:n].to(device)
    adv_images = fgsm_attack(model, images, labels, epsilon, criterion)

    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(3,1,1).to(device)
    std  = torch.tensor([0.2023, 0.1994, 0.2010]).view(3,1,1).to(device)

    with torch.no_grad():
        pred_orig = torch.max(model(images), 1)[1]
        pred_adv  = torch.max(model(adv_images), 1)[1]

    fig, axes = plt.subplots(3, n, figsize=(18, 6))
    row_labels = ['Orijinal', 'Perturbation (x10)', 'Saldirili']

    for i in range(n):
        orig = ((images[i] * std + mean).clamp(0,1)
                .permute(1,2,0).cpu().numpy())
        adv  = ((adv_images[i] * std + mean).clamp(0,1)
                .permute(1,2,0).cpu().numpy())
        pert = np.clip(
            (adv_images[i] - images[i]).permute(1,2,0).cpu().numpy() * 10 + 0.5,
            0, 1
        )

        axes[0,i].imshow(orig)
        axes[1,i].imshow(pert)
        axes[2,i].imshow(adv)

        for row in range(3):
            axes[row,i].axis('off')

        axes[0,i].set_title(f'GT: {classes[labels[i]]}\n'
                             f'Pred: {classes[pred_orig[i]]}',
                             fontsize=7)
        color = 'red' if pred_adv[i] != labels[i] else 'green'
        axes[2,i].set_title(f'Pred: {classes[pred_adv[i]]}',
                             fontsize=7, color=color)

    for row, lbl in enumerate(row_labels):
        axes[row, 0].set_ylabel(lbl, fontsize=9, fontweight='bold')

    plt.suptitle(f'FGSM Saldirisi — epsilon={epsilon}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.savefig(DRIVE_PATH + f'attack_viz_eps{epsilon}.png', dpi=150)
    plt.show()

visualize_attacks(model_a, test_loader, epsilon=0.1, classes=CLASSES, device=device)

In [ ]:
# Grafik 3: Transferability bar chart
scenarios_names = [r[0] for r in transfer_results]
clean_accs      = [r[1] for r in transfer_results]
adv_accs        = [r[2] for r in transfer_results]

x = np.arange(len(scenarios_names))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, clean_accs, width,
               label='Temiz Dogruluk', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + width/2, adv_accs,   width,
               label='Saldiri Altinda', color='tomato',    alpha=0.8)

ax.set_xlabel('Senaryo', fontsize=11)
ax.set_ylabel('Dogruluk (%)', fontsize=11)
ax.set_title('Black-box Transferability Analizi (epsilon=0.1)',
              fontsize=12, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(scenarios_names, fontsize=9)
ax.legend(fontsize=10)
ax.set_ylim(0, 100)
ax.grid(True, alpha=0.3, axis='y')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{bar.get_height():.1f}%', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(DRIVE_PATH + 'transferability.png', dpi=150)
plt.show()

In [ ]:
# Grafik 4: Confusion matrix -- normal vs saldiri altinda
def get_predictions(model, loader, device, epsilon=0):
    criterion = nn.CrossEntropyLoss()
    model.eval()
    all_preds = []
    all_labels = []
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if epsilon > 0:
            images = fgsm_attack(model, images, labels, epsilon, criterion)
        with torch.no_grad():
            _, preds = torch.max(model(images), 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    return all_labels, all_preds

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, eps, title in [
    (axes[0], 0,   'Normal (Saldiri Yok)'),
    (axes[1], 0.1, 'FGSM Saldirisi (epsilon=0.1)'),
]:
    true, pred = get_predictions(model_a, test_loader, device, epsilon=eps)
    cm = confusion_matrix(true, pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASSES, yticklabels=CLASSES,
                ax=ax, cbar=False)
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Tahmin', fontsize=10)
    ax.set_ylabel('Gercek', fontsize=10)
    ax.tick_params(axis='x', rotation=45)

plt.suptitle('Confusion Matrix Karsilastirmasi', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DRIVE_PATH + 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
# Sonuc ozeti
print('=' * 60)
print('PROJE SONUC OZETI')
print('=' * 60)
print(f'\nModel A (SimpleCNN) Baseline  : {history_a["test_acc"][-1]:.2f}%')
print(f'Model B (DeeperCNN) Baseline  : {history_b["test_acc"][-1]:.2f}%')
print(f'Savunmali Model Baseline      : {history_robust["test_acc"][-1]:.2f}%')
print(f'\nFGSM eps=0.1 -- Normal Model  : {fgsm_results[3]:.2f}%')
print(f'FGSM eps=0.1 -- Savunmali     : {robust_fgsm[3]:.2f}%')
print(f'PGD  eps=0.1 -- Normal Model  : {pgd_results[3]:.2f}%')
print(f'PGD  eps=0.1 -- Savunmali     : {robust_pgd[3]:.2f}%')
print(f'\nTransferability (A->B, eps=0.1): {transfer_results[1][3]:.2f}% etki')
print('=' * 60)